<a href="https://colab.research.google.com/github/juanarrow/BigData-notebooks/blob/main/sesion-37.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sesión 37 - Monitorización: Jobtracker, Namenode

**Módulo:** Big Data Aplicado (BDA - 5075)

**Bloque:** RA4 - Monitorización

---

## Objetivos del notebook

1. Simular métricas de un clúster Hadoop con Python
2. Visualizar la evolución temporal de CPU, memoria, disco y red
3. Analizar logs simulados con expresiones regulares
4. Detectar anomalías en las métricas del clúster

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import re
from datetime import datetime, timedelta
import random

random.seed(42)
np.random.seed(42)

print("=== Librerías cargadas ===")
print(f"pandas: {pd.__version__}")
print(f"numpy: {np.__version__}")
print("matplotlib, re, datetime, random")

---

## 1. Simulación de métricas de un clúster Hadoop

Vamos a generar datos realistas de un clúster de 5 nodos durante 7 días,
simulando las métricas que veríamos en las interfaces web del Namenode y ResourceManager.

In [ ]:
NODOS = ['namenode', 'datanode-01', 'datanode-02', 'datanode-03', 'datanode-04']
HORAS = 7 * 24  # 7 días
INTERVALO_MIN = 15  # Muestreo cada 15 minutos

inicio = datetime(2026, 3, 1, 0, 0)
timestamps = [inicio + timedelta(minutes=i * INTERVALO_MIN) for i in range(HORAS * 4)]

registros = []

for nodo in NODOS:
    # Línea base distinta para cada nodo
    cpu_base = 30 if nodo == 'namenode' else np.random.randint(20, 45)
    mem_base = 60 if nodo == 'namenode' else np.random.randint(40, 55)
    disk_base = 45 if nodo == 'namenode' else np.random.randint(50, 70)
    net_base = 200 if nodo == 'namenode' else np.random.randint(100, 300)

    for i, ts in enumerate(timestamps):
        hora_dia = ts.hour

        # Patrón diurno: más carga de 9 a 18h
        factor_diurno = 1.3 if 9 <= hora_dia <= 18 else 0.7

        # Simular pico de carga (job MapReduce grande) en día 4
        factor_pico = 1.0
        if 4 * 96 <= i <= 4 * 96 + 24:  # ~6 horas del día 4
            factor_pico = 1.8 if nodo != 'namenode' else 1.4

        # Simular problema en datanode-03 en día 5
        factor_problema = 1.0
        if nodo == 'datanode-03' and 5 * 96 <= i <= 5 * 96 + 48:
            factor_problema = 2.2

        ruido = np.random.normal(0, 3)

        cpu = min(100, max(0, cpu_base * factor_diurno * factor_pico * factor_problema + ruido))
        mem = min(100, max(0, mem_base * factor_diurno * factor_pico * factor_problema + ruido * 0.5))
        disk = min(100, max(0, disk_base + (i / len(timestamps)) * 10 + ruido * 0.3))
        net_mbps = max(0, net_base * factor_diurno * factor_pico * factor_problema + ruido * 10)

        registros.append({
            'timestamp': ts,
            'nodo': nodo,
            'cpu_pct': round(cpu, 1),
            'memoria_pct': round(mem, 1),
            'disco_pct': round(disk, 1),
            'red_mbps': round(net_mbps, 1)
        })

df_metricas = pd.DataFrame(registros)

print(f"Total de registros generados: {len(df_metricas):,}")
print(f"Periodo: {df_metricas['timestamp'].min()} → {df_metricas['timestamp'].max()}")
print(f"Nodos: {df_metricas['nodo'].nunique()}")
print()
df_metricas.head(10)

---

## 2. Dashboard de monitorización: visualización de métricas

Creamos un dashboard similar al que veríamos en el ResourceManager o una herramienta como Grafana.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('Dashboard de Monitorización del Clúster Hadoop', fontsize=16, fontweight='bold')

metricas = [
    ('cpu_pct', 'CPU (%)', axes[0, 0], 85),
    ('memoria_pct', 'Memoria (%)', axes[0, 1], 90),
    ('disco_pct', 'Disco (%)', axes[1, 0], 80),
    ('red_mbps', 'Red (Mbps)', axes[1, 1], 500),
]

colores = {'namenode': '#e74c3c', 'datanode-01': '#3498db',
           'datanode-02': '#2ecc71', 'datanode-03': '#f39c12',
           'datanode-04': '#9b59b6'}

for col, titulo, ax, umbral in metricas:
    for nodo in NODOS:
        datos_nodo = df_metricas[df_metricas['nodo'] == nodo]
        ax.plot(datos_nodo['timestamp'], datos_nodo[col],
                label=nodo, color=colores[nodo], alpha=0.7, linewidth=0.8)

    ax.axhline(y=umbral, color='red', linestyle='--', alpha=0.5, label=f'Umbral ({umbral})')
    ax.set_title(titulo, fontsize=12, fontweight='bold')
    ax.set_xlabel('Fecha')
    ax.set_ylabel(titulo)
    ax.legend(fontsize=7, loc='upper left')
    ax.grid(True, alpha=0.3)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%d/%m'))
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

print("📊 Observa:")
print("   - El pico de carga del día 4 (job MapReduce grande)")
print("   - El problema en datanode-03 el día 5")
print("   - El patrón diurno (más carga de 9h a 18h)")
print("   - El crecimiento gradual del disco")

---

## 3. Detección de anomalías con umbrales

Una tarea fundamental de la monitorización es detectar cuándo las métricas superan
umbrales definidos. Vamos a implementar un sistema sencillo de alertas.

In [ ]:
UMBRALES = {
    'cpu_pct': {'warning': 70, 'critical': 85},
    'memoria_pct': {'warning': 75, 'critical': 90},
    'disco_pct': {'warning': 75, 'critical': 85},
    'red_mbps': {'warning': 450, 'critical': 600}
}

alertas = []

for _, fila in df_metricas.iterrows():
    for metrica, niveles in UMBRALES.items():
        valor = fila[metrica]
        if valor >= niveles['critical']:
            alertas.append({
                'timestamp': fila['timestamp'],
                'nodo': fila['nodo'],
                'metrica': metrica,
                'valor': valor,
                'nivel': 'CRITICAL'
            })
        elif valor >= niveles['warning']:
            alertas.append({
                'timestamp': fila['timestamp'],
                'nodo': fila['nodo'],
                'metrica': metrica,
                'valor': valor,
                'nivel': 'WARNING'
            })

df_alertas = pd.DataFrame(alertas)

print("=" * 60)
print("RESUMEN DE ALERTAS DETECTADAS")
print("=" * 60)
print(f"\nTotal de alertas: {len(df_alertas)}")
print(f"\nPor nivel:")
print(df_alertas['nivel'].value_counts().to_string())
print(f"\nPor nodo:")
print(df_alertas.groupby(['nodo', 'nivel']).size().unstack(fill_value=0).to_string())
print(f"\nPor métrica:")
print(df_alertas.groupby(['metrica', 'nivel']).size().unstack(fill_value=0).to_string())

print("\n⚠️  ¿Qué nodo tiene más alertas CRITICAL? ¿Coincide con lo que vimos en el dashboard?")

---

## 4. Análisis de logs de Hadoop con expresiones regulares

En la realidad, los logs de Hadoop siguen un formato Log4j estándar.
Vamos a simular logs y analizarlos con `re`.

In [ ]:
plantillas_log = [
    ("INFO", "org.apache.hadoop.hdfs.server.datanode.DataNode",
     "Received block blk_{blk_id} of size {size} from /{ip}"),
    ("INFO", "org.apache.hadoop.hdfs.server.namenode.FSNamesystem",
     "BLOCK* addStoredBlock: blk_{blk_id} is added to {nodo}"),
    ("WARN", "org.apache.hadoop.hdfs.server.blockmanagement.DatanodeManager",
     "Node {nodo}:50010 is slow. Response time: {tiempo}ms"),
    ("WARN", "org.apache.hadoop.hdfs.server.namenode.FSNamesystem",
     "Under-replicated block blk_{blk_id}: {repl} replicas instead of 3"),
    ("ERROR", "org.apache.hadoop.hdfs.server.datanode.DataNode",
     "IOException in offerService: connection refused to {nodo}:50010"),
    ("ERROR", "org.apache.hadoop.mapreduce.v2.app.job.impl.TaskAttemptImpl",
     "Container killed by YARN for exceeding memory limits. {mem}MB of {max_mem}MB used"),
    ("FATAL", "org.apache.hadoop.hdfs.server.datanode.DataNode",
     "Disk error on /data/{disco}: No space left on device"),
    ("INFO", "org.apache.hadoop.yarn.server.resourcemanager.rmapp.RMAppImpl",
     "Application application_{app_id} completed with status: {status}"),
]

def generar_logs(n_lineas=200):
    logs = []
    base_time = datetime(2026, 3, 5, 8, 0, 0)

    for i in range(n_lineas):
        ts = base_time + timedelta(seconds=random.randint(0, 36000))
        nivel, clase, plantilla = random.choices(
            plantillas_log,
            weights=[30, 25, 10, 8, 5, 5, 2, 15],
            k=1
        )[0]

        mensaje = plantilla.format(
            blk_id=random.randint(1000000, 9999999),
            size=random.randint(64_000_000, 256_000_000),
            ip=f"192.168.1.{random.randint(10, 50)}",
            nodo=random.choice(['datanode-01', 'datanode-02', 'datanode-03', 'datanode-04']),
            tiempo=random.randint(100, 5000),
            repl=random.choice([1, 2]),
            mem=random.randint(2048, 8192),
            max_mem=random.choice([2048, 4096]),
            disco=f"disk{random.randint(1, 4)}",
            app_id=f"1709_{random.randint(1000, 9999)}",
            status=random.choice(['FINISHED', 'FINISHED', 'FINISHED', 'FAILED', 'KILLED'])
        )

        linea = f"{ts.strftime('%Y-%m-%d %H:%M:%S')} {nivel} {clase}: {mensaje}"
        logs.append(linea)

    return sorted(logs)

logs_simulados = generar_logs(200)

print(f"Se han generado {len(logs_simulados)} líneas de log\n")
print("=== Primeras 10 líneas ===")
for linea in logs_simulados[:10]:
    print(linea)

In [ ]:
patron_log = re.compile(
    r'(\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}) (INFO|WARN|ERROR|FATAL) ([\w.]+): (.+)'
)

parsed_logs = []
for linea in logs_simulados:
    match = patron_log.match(linea)
    if match:
        parsed_logs.append({
            'timestamp': match.group(1),
            'nivel': match.group(2),
            'clase': match.group(3),
            'mensaje': match.group(4)
        })

df_logs = pd.DataFrame(parsed_logs)
df_logs['timestamp'] = pd.to_datetime(df_logs['timestamp'])

print("=" * 60)
print("ANÁLISIS DE LOGS")
print("=" * 60)

print("\n📊 Distribución por nivel de severidad:")
conteo_niveles = df_logs['nivel'].value_counts()
for nivel, cantidad in conteo_niveles.items():
    barra = '█' * (cantidad // 2)
    print(f"  {nivel:6s}: {cantidad:3d} {barra}")

print("\n🔴 Errores y fatales encontrados:")
errores = df_logs[df_logs['nivel'].isin(['ERROR', 'FATAL'])]
for _, row in errores.iterrows():
    print(f"  [{row['nivel']}] {row['timestamp']} - {row['mensaje'][:80]}")

print(f"\n⚠️  Total de problemas (ERROR + FATAL): {len(errores)}")

---

## 5. Análisis temporal de logs

Agrupamos los logs por hora para detectar si hay periodos con más errores.

In [ ]:
df_logs['hora'] = df_logs['timestamp'].dt.hour

logs_por_hora = df_logs.groupby(['hora', 'nivel']).size().unstack(fill_value=0)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

colores_nivel = {'INFO': '#2ecc71', 'WARN': '#f39c12', 'ERROR': '#e74c3c', 'FATAL': '#8e44ad'}
columnas_presentes = [c for c in ['INFO', 'WARN', 'ERROR', 'FATAL'] if c in logs_por_hora.columns]

logs_por_hora[columnas_presentes].plot(
    kind='bar', stacked=True, ax=ax1,
    color=[colores_nivel[c] for c in columnas_presentes]
)
ax1.set_title('Distribución de logs por hora y severidad', fontweight='bold')
ax1.set_xlabel('Hora del día')
ax1.set_ylabel('Número de entradas')
ax1.legend(title='Nivel')
ax1.grid(True, alpha=0.3, axis='y')

niveles_conteo = df_logs['nivel'].value_counts()
ax2.pie(niveles_conteo.values,
        labels=niveles_conteo.index,
        colors=[colores_nivel.get(n, '#bdc3c7') for n in niveles_conteo.index],
        autopct='%1.1f%%', startangle=90)
ax2.set_title('Proporción por nivel de severidad', fontweight='bold')

plt.tight_layout()
plt.show()

print("📊 ¿Hay alguna hora con concentración inusual de errores?")

---

## 6. Ejercicios propuestos

### Ejercicio 1: Análisis de tendencias
Calcula la media móvil (rolling mean) de CPU para cada nodo usando una ventana de 24 muestras
(6 horas) y dibuja el resultado. ¿Se suaviza el ruido?

### Ejercicio 2: Correlación entre métricas
¿Existe correlación entre el uso de CPU y el tráfico de red en los DataNodes?
Calcula el coeficiente de correlación y crea un scatter plot.

### Ejercicio 3: Extracción avanzada de logs
Usando expresiones regulares, extrae del log:
- Todas las IPs que aparecen en los mensajes
- Todos los IDs de bloque (`blk_XXXXXXX`)
- Los nodos que presentan errores de conexión

### Ejercicio 4: Sistema de alertas mejorado
Implementa un sistema que detecte no solo umbrales estáticos sino también:
- Incrementos bruscos (valor actual > 1.5 × media de la última hora)
- Tendencia creciente sostenida durante más de 3 horas

In [ ]:
# EJERCICIO 1: Media móvil de CPU
# Completa el código

fig, ax = plt.subplots(figsize=(14, 5))

for nodo in NODOS:
    datos_nodo = df_metricas[df_metricas['nodo'] == nodo].copy()
    datos_nodo['cpu_rolling'] = datos_nodo['cpu_pct'].rolling(window=24, center=True).mean()

    ax.plot(datos_nodo['timestamp'], datos_nodo['cpu_pct'],
            alpha=0.2, color=colores[nodo])
    ax.plot(datos_nodo['timestamp'], datos_nodo['cpu_rolling'],
            label=f'{nodo} (media 6h)', color=colores[nodo], linewidth=2)

ax.set_title('CPU por nodo - Media móvil de 6 horas', fontweight='bold')
ax.set_xlabel('Fecha')
ax.set_ylabel('CPU (%)')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%d/%m %Hh'))
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

print("✅ La media móvil suaviza el ruido y permite ver tendencias claras.")
print("   Observa cómo se destaca el pico del día 4 y el problema del día 5.")

---

## Resumen

En este notebook hemos aprendido a:

| Concepto | Herramienta Python | Aplicación |
|----------|-------------------|------------|
| Generar métricas simuladas | `random`, `numpy` | Simular clúster Hadoop |
| Visualizar métricas | `matplotlib` | Dashboards de monitorización |
| Detectar anomalías | `pandas` + umbrales | Alertas automáticas |
| Analizar logs | `re` (regex) | Parsear logs Log4j |
| Análisis temporal | `pandas.groupby` | Patrones por hora/día |
| Tendencias | `pandas.rolling` | Media móvil para suavizar |

### Próximo paso

En la **Sesión 39** veremos **Ganglia**, una herramienta profesional de monitorización
de clústeres que automatiza todo lo que hemos hecho manualmente aquí.

---

*Recuerda completar los ejercicios propuestos antes de la sesión 39.*